# Road Network Connectivity Analysis

This notebook analyzes road network connectivity between urban centers, quantifying the impact of unpaved roads on inter-city accessibility.

## 1. Configuration

In [ ]:
from pathlib import Path

# Base directories
BASE_DATA_DIR = Path("Pat/to/Road_Surface_Paper_Data_Planet")

# Input data
PARQUET_PATHS = {
    2024: BASE_DATA_DIR / "road_surfaces" / "2024" / "all_roads_final.parquet",
    2020: BASE_DATA_DIR / "road_surfaces" / "2020" / "all_roads_final.parquet",
    2016: BASE_DATA_DIR / "road_surfaces" / "2016" / "all_roads_final.parquet",
}

URBAN_CENTERS_GPKG = BASE_DATA_DIR / "GHS_UCDB_GLOBE_R2024A.gpkg"
WORLD_SHAPEFILE = BASE_DATA_DIR / "ne_110m_admin_0_countries.shp"
GLOBAL_PBF = BASE_DATA_DIR / "planet_major_roads.pbf"

# Output directories
OUTPUT_DIR = Path("./output")
EXTRACTS_DIR = OUTPUT_DIR / "country_extracts"
RESULTS_CSV = OUTPUT_DIR / "connectivity_results.csv"
CONNECTIVITY_GPKG = OUTPUT_DIR / "maps" / "connectivity.gpkg"

for d in [OUTPUT_DIR, EXTRACTS_DIR, OUTPUT_DIR / "maps"]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration loaded")

## 2. Imports

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import geopandas as gpd
import duckdb
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.cluster import KMeans

## 3. Data Preparation

### 3.1 Load World Boundaries

In [ ]:
world = gpd.read_file(str(WORLD_SHAPEFILE))
world = world.assign(area=world.geometry.to_crs(3857).area).sort_values("area")
print(f"Loaded {len(world)} countries")

### 3.2 Identify Unpaved Roads

In [ ]:
con = duckdb.connect()
df_unpaved = con.execute(f"""
    SELECT osm_id FROM read_parquet('{PARQUET_PATHS[2024]}')
    WHERE road_class = 'unpaved'
""").fetchdf()

unpaved_osmids = set(int(i.split("/")[1]) for i in df_unpaved.osm_id)
print(f"Found {len(unpaved_osmids):,} unpaved roads")

## 4. Analysis Functions

### 4.1 Entry Point Detection

In [ ]:
def reduce_entry_points(points, k=5):
    """Cluster entry points to k representatives using K-Means."""
    if len(points) <= k:
        return points
    coords = np.array([[p.x, p.y] for p in points])
    kmeans = KMeans(n_clusters=k, n_init='auto', random_state=42).fit(coords)
    return [points[np.argmin(np.linalg.norm(coords - c, axis=1))] for c in kmeans.cluster_centers_]


def get_entry_nodes(urban_geom, edges_gdf, graph, n=3):
    """Find network nodes where major roads cross urban boundary."""
    major = edges_gdf[~edges_gdf["highway"].isin(["tertiary", "tertiary_link"])]
    boundary = urban_geom.boundary
    
    pts = []
    for geom in major[major.intersects(urban_geom)].geometry:
        inter = geom.intersection(boundary)
        if inter.geom_type == 'Point':
            pts.append(inter)
        elif hasattr(inter, 'geoms'):
            pts.extend(p for p in inter.geoms if p.geom_type == 'Point')
    
    pts = reduce_entry_points(pts, n)
    nodes = set()
    for pt in pts:
        try:
            nodes.add(ox.distance.nearest_nodes(graph, X=pt.x, Y=pt.y))
        except:
            pass
    return list(nodes)

### 4.2 Connectivity Matrix

In [ ]:
def compute_connectivity_matrix(urban_gdf, G, edges_gdf):
    """Compute pairwise shortest paths between urban centers."""
    ids = urban_gdf.index.tolist()
    matrix = pd.DataFrame(index=ids, columns=ids, dtype=float)
    
    entry = {i: get_entry_nodes(urban_gdf.loc[i].geometry, edges_gdf, G) 
             for i in tqdm(ids, desc="Entry nodes")}
    
    for i in tqdm(ids, desc="Shortest paths"):
        for j in ids:
            if i == j:
                matrix.loc[i, j] = 0
                continue
            
            min_d = float("inf")
            for u in entry.get(i, []):
                for v in entry.get(j, []):
                    try:
                        min_d = min(min_d, nx.shortest_path_length(G, u, v, weight="length"))
                    except:
                        pass
            matrix.loc[i, j] = min_d if min_d < float("inf") else np.nan
    
    return matrix


def is_unpaved(osmids, unpaved_set):
    """Check if edge is unpaved."""
    if isinstance(osmids, list):
        return any(o in unpaved_set for o in osmids)
    return osmids in unpaved_set

## 5. Main Analysis Loop

In [ ]:
urban_centers = gpd.read_file(str(URBAN_CENTERS_GPKG)).to_crs(4326)

results = pd.read_csv(str(RESULTS_CSV)).to_dict('records') if RESULTS_CSV.exists() else []
done = {r['country'] for r in results}

for idx, row in world.iterrows():
    cc = row.get('SOV_A3', f"r{idx}")
    if cc in done:
        continue
    
    print(f"\nProcessing {cc}")
    
    uc = urban_centers[urban_centers.intersects(row.geometry)].copy()
    if uc.empty:
        print("  No urban centers")
        continue
    
    osm_file = EXTRACTS_DIR / f"{cc}.osm"
    if not osm_file.exists():
        continue
    
    try:
        G = ox.graph_from_xml(str(osm_file))
    except Exception as e:
        print(f"  Load error: {e}")
        continue
    
    G.remove_edges_from([(u,v,k) for u,v,k,d in G.edges(keys=True, data=True) 
                         if d.get("highway") in ["tertiary", "tertiary_link"]])
    
    edges = ox.graph_to_gdfs(G, nodes=False)
    uc["geometry"] = uc.geometry.convex_hull
    
    mat_full = compute_connectivity_matrix(uc, G, edges)
    
    G.remove_edges_from([(u,v,k) for u,v,k,d in G.edges(keys=True, data=True) 
                         if is_unpaved(d.get("osmid"), unpaved_osmids)])
    edges_r = ox.graph_to_gdfs(G, nodes=False)
    
    mat_rest = compute_connectivity_matrix(uc, G, edges_r)
    
    delta = ((mat_rest - mat_full) / mat_full).values.flatten()
    delta = delta[np.isfinite(delta)]
    
    full_nan = mat_full.isna().values
    rest_nan = mat_rest.isna().values
    
    comps = list(nx.weakly_connected_components(G))
    
    results.append({
        "country": cc,
        "n_urban": len(uc),
        "mean_rel_change": np.mean(delta) if len(delta) else np.nan,
        "increased_ratio": np.mean(delta > 0) if len(delta) else np.nan,
        "unreachable_full": full_nan.sum(),
        "unreachable_rest": rest_nan.sum(),
        "new_unreachable": (rest_nan & ~full_nan).sum() / mat_full.size,
        "n_components": len(comps),
        "largest_comp_frac": len(max(comps, key=len)) / len(G),
    })
    
    pd.DataFrame(results).to_csv(str(RESULTS_CSV), index=False)
    print(f"  Mean change: {results[-1]['mean_rel_change']:.1%}")

print("\nComplete!")

## 6. Results

In [ ]:
df = pd.read_csv(str(RESULTS_CSV))

gdf = world.merge(df, left_on="SOV_A3", right_on="country")
gdf.to_file(str(CONNECTIVITY_GPKG), layer="connectivity", driver="GPKG")

print(f"Countries analyzed: {len(df)}")
print(f"Mean distance increase: {df['mean_rel_change'].mean():.1%}")
print(f"Mean new unreachable: {df['new_unreachable'].mean():.1%}")
print(f"\nTop 5 most impacted:")
print(df.nlargest(5, 'mean_rel_change')[['country', 'mean_rel_change', 'new_unreachable']])